In [ ]:
# Instação
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import month, col
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, VectorAssembler, Normalizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from google.colab import files

In [ ]:
# Upload do arquivo
uploaded = files.upload()

Saving videos-comments-tratados.snappy.parquet to videos-comments-tratados.snappy.parquet


In [ ]:
# Criação da sessão
spark = SparkSession.builder \
    .appName("PreparacaoRedesSociais") \
    .getOrCreate()

In [ ]:
# Leitura do data frame
df_video = spark.read.parquet("videos-comments-tratados.snappy.parquet")

df_video.printSchema()
print("Quantidade de registros:", df_video.count())
df_video.show(5, truncate=False)

root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)

Quantidade de registros: 18409
+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Converção
df_video = df_video.withColumn("Published At", col("Published At").cast("timestamp"))
df_video = df_video.withColumn("Month", month(col("Published At")))

df_video.select("Published At", "Month").show(5)

+-------------------+-----+
|       Published At|Month|
+-------------------+-----+
|2022-08-23 00:00:00|    8|
|2022-08-23 00:00:00|    8|
|2022-08-23 00:00:00|    8|
|2022-08-23 00:00:00|    8|
|2022-08-23 00:00:00|    8|
+-------------------+-----+
only showing top 5 rows


In [ ]:
# converção para numeros
indexer = StringIndexer(
    inputCol="keyword",
    outputCol="Keyword Index",
    handleInvalid="keep"
)

indexer_model = indexer.fit(df_video)
df_video = indexer_model.transform(df_video)

df_video.select("keyword", "Keyword Index").show(5)

+-------+-------------+
|keyword|Keyword Index|
+-------+-------------+
|   tech|         17.0|
|   tech|         17.0|
|   tech|         17.0|
|   tech|         17.0|
|   tech|         17.0|
+-------+-------------+
only showing top 5 rows


In [ ]:
# Preparação
df_video = df_video.withColumn("Likes", col("Likes").cast(DoubleType()))
df_video = df_video.withColumn("Views", col("Views").cast(DoubleType()))
df_video = df_video.withColumn("Year", col("Year").cast(DoubleType()))
df_video = df_video.withColumn("Month", col("Month").cast(DoubleType()))
df_video = df_video.withColumn("Keyword Index", col("Keyword Index").cast(DoubleType()))
df_video = df_video.withColumn("Comments", col("Comments").cast(DoubleType()))

df_video = df_video.fillna({
    "Likes": 0.0,
    "Views": 0.0,
    "Year": 0.0,
    "Month": 0.0,
    "Keyword Index": 0.0,
    "Comments": 0.0
})

In [ ]:
#Criação vetorial
assembler = VectorAssembler(
    inputCols=["Likes", "Views", "Year", "Month", "Keyword Index"],
    outputCol="Features"
)

df_video = assembler.transform(df_video)

df_video.select("Features").show(5, truncate=False)

+---------------------------------+
|Features                         |
+---------------------------------+
|[3407.0,135612.0,2022.0,8.0,17.0]|
|[3407.0,135612.0,2022.0,8.0,17.0]|
|[3407.0,135612.0,2022.0,8.0,17.0]|
|[3407.0,135612.0,2022.0,8.0,17.0]|
|[3407.0,135612.0,2022.0,8.0,17.0]|
+---------------------------------+
only showing top 5 rows


In [ ]:
#valores para comparação
normalizer = Normalizer(
    inputCol="Features",
    outputCol="Features Normal",
    p=2.0
)

df_video = normalizer.transform(df_video)

df_video.select("Features", "Features Normal").show(5, truncate=False)

+---------------------------------+--------------------------------------------------------------------------------------------------------+
|Features                         |Features Normal                                                                                         |
+---------------------------------+--------------------------------------------------------------------------------------------------------+
|[3407.0,135612.0,2022.0,8.0,17.0]|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|
|[3407.0,135612.0,2022.0,8.0,17.0]|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|
|[3407.0,135612.0,2022.0,8.0,17.0]|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|
|[3407.0,135612.0,2022.0,8.0,17.0]|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|
|[3407.0,1356

In [ ]:
# Redução
pca = PCA(
    k=1,
    inputCol="Features Normal",
    outputCol="Features PCA"
)

pca_model = pca.fit(df_video)
df_video = pca_model.transform(df_video)

df_video.select("Features Normal", "Features PCA").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------+---------------------+
|Features Normal                                                                                         |Features PCA         |
+--------------------------------------------------------------------------------------------------------+---------------------+
|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|[0.39522435928489963]|
|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|[0.39522435928489963]|
|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|[0.39522435928489963]|
|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.2530417548674107E-4]|[0.39522435928489963]|
|[0.02511243093431334,0.9995735203592899,0.014903826049070024,5.896667081728991E-5,1.253041754867

In [ ]:
# divisão do DF
treino, teste = df_video.randomSplit([0.8, 0.2], seed=42)

print("Treino:", treino.count())
print("Teste:", teste.count())

Treino: 14789
Teste: 3620


In [ ]:
#Criação de modelo linear
lr = LinearRegression(
    featuresCol="Features Normal",
    labelCol="Comments",
    predictionCol="prediction"
)

modelo_lr = lr.fit(treino)

In [ ]:
#Previsão
predicoes = modelo_lr.transform(teste)

predicoes.select("Comments", "prediction").show(10, truncate=False)

+--------+-----------------+
|Comments|prediction       |
+--------+-----------------+
|18860.0 |9647.170910602552|
|18860.0 |9647.170910602552|
|18860.0 |9647.170910602552|
|4853.0  |9744.51796757942 |
|4853.0  |9744.51796757942 |
|2347.0  |8668.763187340985|
|2347.0  |8668.763187340985|
|17264.0 |9844.300527502375|
|525.0   |9535.895106612297|
|525.0   |9535.895106612297|
+--------+-----------------+
only showing top 10 rows


In [ ]:
#Metricas
evaluator_rmse = RegressionEvaluator(
    labelCol="Comments",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="Comments",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="Comments",
    predictionCol="prediction",
    metricName="r2"
)

print("===== Avaliação =====")
print("RMSE:", evaluator_rmse.evaluate(predicoes))
print("MAE:", evaluator_mae.evaluate(predicoes))
print("R2:", evaluator_r2.evaluate(predicoes))

===== Avaliação =====
RMSE: 43345.23343236093
MAE: 11982.35655803593
R2: 0.008252984288786291


In [ ]:
#Salvar o arquivo
df_video.write.mode("overwrite").parquet("videos-preparados-parquet")

print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!
